In [ ]:
import pandas as pd
import numpy as np

# 1. Load the original dataset
df = pd.read_csv("AB_NYC_2019.csv")

# -----------------------------------
# 2. Text Normalization 
# -----------------------------------
# Standardizing text to lowercase to ensure consistency and improve grouping
df['name'] = df['name'].str.lower()
df['host_name'] = df['host_name'].str.lower()
df['neighbourhood'] = df['neighbourhood'].str.lower()

# -----------------------------------
# 3. Handling Missing Values and Duplicates
# -----------------------------------
# Removing duplicate rows
df.drop_duplicates(inplace=True)

# Dropping rows where essential information (listing or host name) is missing
df.dropna(subset=['name', 'host_name'], inplace=True)

# Filling missing values in 'reviews_per_month' with 0 as no reviews were recorded
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# -----------------------------------
# 4. Data Cleaning (Logical Filters)
# -----------------------------------
# Keeping realistic prices (between $1 and $1000)
df = df[(df['price'] > 0) & (df['price'] < 1000)]

# Removing listings with unrealistic 'minimum_nights' (more than a year)
df = df[df['minimum_nights'] < 365]

# -----------------------------------
# 5. Outlier Removal using IQR Method
# -----------------------------------
# Calculating the Interquartile Range (IQR) for the price column
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtering data to keep only values within the non-outlier range
df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]

# -----------------------------------
# 6. Final Formatting and Export
# -----------------------------------
# Dropping columns that are not useful for general data analysis
cols_to_drop = ['id', 'host_id', 'last_review']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Resetting the index to maintain a clean numerical order
df.reset_index(drop=True, inplace=True)

# Save the finalized cleaned dataset
df.to_csv("airbnb_cleaned.csv", index=False)

print("Final dataset is ready!")
print(f"Cleaned Data Shape: {df.shape}")
print(df.head())